# 02 · Text Preprocessing
**Project:** Tweet Emotion Detection · **Input:** raw CSVs in `data/` · **Output:** `data/processed/`

**Goal.** Turn noisy raw tweets into clean, consistent tokens **without throwing away emotion signals**.
Every step below is justified by a finding from `01_eda.ipynb` (referenced as *EDA Q#*).

**Scope.** Only preprocessing. Turning tokens into numbers (BoW, TF-IDF, embeddings) is the next notebook, `03_representation`.

### Pipeline

| Step | Operation | Why (EDA evidence) |
|---|---|---|
| 1 | Remove duplicate tweets (train only) | 25 spam copies of one joy tweet (Q2) |
| 2 | Fix encoding noise: literal `\n`, HTML entities, Unicode, curly quotes, invisible characters | 279 `\n`, 240 `&amp;`, 43 `’` (Q2) |
| 3 | Remove `@user` mentions and URLs | `@user` in 44% of tweets, carries no meaning (Q5) |
| 4 | Hashtags: drop `#`, keep and split the word | Hashtags often name the emotion (`#angry`, `#sad`) (Q6) |
| 5 | Emojis → one word-token each (😭 → `loudly_crying_face`) | Emojis differ strongly by emotion (😡 vs 😭) (Q5–Q6) |
| 6 | Lowercase + expand contractions | Unify word forms; make negation explicit (`can't` → `can not`) (Q9) |
| 7 | Tokenize with `TweetTokenizer` (also shortens `sooo` → `soo`) | Tweet-aware tokenizer (Q5) |
| 8 | Remove punctuation and numbers, **keep `!` and `?`** | `!` is typical of joy, `?` of anger (Q5) |
| 9 | Stop-word removal **without negations** *(variant)* | Negation in 26% of tweets (Q9) |
| 10 | Lemmatization with POS tags *(variant)* | Merge word forms (*crying, cried* → *cry*) |


## Setup

In [ ]:
import os, re, html, unicodedata, warnings
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer

for resource in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

DATA_DIR, OUT_DIR, FIG_DIR = "../data", "../data/processed", "../results/figures/preprocessing"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
LABELS = ["anger", "joy", "optimism", "sadness"]
PARTS = ["train", "validation", "test"]